# MD-GRTN — No Diffusion, Single Sequence (Clean Build)

**Pipeline:** X_RecN → MAF → MGRC (GCN+GRU ×3) → Spatial Transformer ×5 → Temporal Transformer ×5 → Output

**Key fix:** Raw data stats computed before normalization → correct denorm → no MAE explosion

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} ✓')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    data_path    = "/content/PEMS08.npz"
    num_nodes    = 170
    in_features  = 3
    seq_len      = 12
    pred_len     = 12
    feature_idx  = 0       # flow channel

    # model
    d_model      = 96      # 96 ÷ 3 = 32 ✓
    n_heads      = 3
    num_layers   = 5       # spatial + temporal transformer layers each
    gru_layers   = 3       # stacked GCN+GRU layers
    dropout      = 0.1

    # training
    batch_size   = 64
    lr           = 1e-3
    weight_decay = 0.01
    epochs       = 100
    patience     = 15
    delta_h      = 1.0
    train_ratio  = 0.7
    val_ratio    = 0.1

    ckpt_path = 'mdgrtn_nodiff_ckpt.pt'
    best_path = 'mdgrtn_nodiff_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert cfg.d_model % cfg.n_heads == 0
print(f'Config | d_model={cfg.d_model} n_heads={cfg.n_heads} head_dim={cfg.d_model//cfg.n_heads} ✓')
print(f'num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers} | device={device}')

In [ ]:
# ══════════════════════════════════════════════════
# DATA — correct normalization pipeline
# 1. Load raw data
# 2. Compute mean/std on RAW data → real vehicle counts
# 3. Normalize for model input
# 4. mean_flow/std_flow used for denorm in evaluation
# ══════════════════════════════════════════════════

# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

raw      = np.load(cfg.data_path)
print('Keys:', list(raw.keys()))
data_raw = raw['data'].astype(np.float32)   # (T, N, F)
T, N, F  = data_raw.shape
print(f'Raw shape: {data_raw.shape}')       # (17856, 170, 3)

# Stats computed on RAW data (real vehicle counts)
mean_np  = data_raw.mean(axis=0)            # (N, F)
std_np   = data_raw.std(axis=0) + 1e-8     # (N, F)
print(f'mean_np shape: {mean_np.shape}')    # (170, 3)

# Normalize
data_norm = (data_raw - mean_np[None]) / std_np[None]   # (T, N, F)
print(f'Normalized range: {data_norm.min():.2f} to {data_norm.max():.2f}')

# Denorm tensors — per-sensor flow channel
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).float().to(device)  # (170,)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).float().to(device)
print(f'mean_flow: min={mean_flow.min():.1f}  max={mean_flow.max():.1f}  (real vehicle counts)')
print(f'std_flow:  min={std_flow.min():.1f}   max={std_flow.max():.1f}')

# Adjacency matrix
for key in ('adjacency_matrix','adj_mx','adj'):
    if key in raw:
        A_dist_np = raw[key].astype(np.float32)
        print(f'Adjacency from "{key}" shape={A_dist_np.shape}')
        break
else:
    print('No adjacency — using identity')
    A_dist_np = np.eye(N, dtype=np.float32)

deg       = A_dist_np.sum(1, keepdims=True) + 1e-8
A_dist_np = A_dist_np / deg
A_dist    = torch.from_numpy(A_dist_np).to(device)

print('Data loaded ✓')

In [ ]:
class TrafficDataset(Dataset):
    def __init__(self, data, seq_len, pred_len, feature_idx):
        self.data   = torch.from_numpy(data).float()
        self.S      = seq_len
        self.P      = pred_len
        self.fi     = feature_idx
        self.length = len(data) - seq_len - pred_len + 1

    def __len__(self): return self.length

    def __getitem__(self, i):
        x = self.data[i        : i+self.S]              # (S, N, F)
        y = self.data[i+self.S : i+self.S+self.P, :, self.fi]  # (P, N)
        return x, y


set_seed()
t1 = int(T * cfg.train_ratio)
t2 = int(T * (cfg.train_ratio + cfg.val_ratio))

kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
dl_train = DataLoader(TrafficDataset(data_norm[:t1],  cfg.seq_len, cfg.pred_len, cfg.feature_idx), shuffle=True,  **kw)
dl_val   = DataLoader(TrafficDataset(data_norm[t1:t2],cfg.seq_len, cfg.pred_len, cfg.feature_idx), shuffle=False, **kw)
dl_test  = DataLoader(TrafficDataset(data_norm[t2:],  cfg.seq_len, cfg.pred_len, cfg.feature_idx), shuffle=False, **kw)

print(f'Train={len(dl_train.dataset)} | Val={len(dl_val.dataset)} | Test={len(dl_test.dataset)}')
print(f'Split: {cfg.train_ratio*100:.0f}/{cfg.val_ratio*100:.0f}/{(1-cfg.train_ratio-cfg.val_ratio)*100:.0f}')

# ── Sanity check ──
x_s, y_s = next(iter(dl_val))
y_real    = y_s * std_flow[None,None,:].cpu() + mean_flow[None,None,:].cpu()
print(f'y normalized : {y_s.min():.2f} to {y_s.max():.2f}')
print(f'y real units : {y_real.min():.1f} to {y_real.max():.1f}  ← should be 0-500 vehicles ✓')

In [ ]:
# ══════════════════════════════════════════════════
# MAF — direct attention, NO BackNet/diffusion
# x → W_Q, W_K, W_V → scaled dot-product → FC
# ══════════════════════════════════════════════════

class MAFModule(nn.Module):
    def __init__(self, in_dim, d_model, n_heads, n_seqs=1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_j  = d_model // n_heads
        self.W_Q  = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_K  = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_V  = nn.ModuleList([nn.Linear(in_dim, d_model, bias=False) for _ in range(n_seqs)])
        self.W_MH = nn.Linear(d_model * n_seqs, d_model)

    def forward(self, seqs):
        B, S, N, _ = seqs[0].shape
        heads = []
        for i, x in enumerate(seqs):
            f  = x.reshape(B*S, N, -1)
            Q  = self.W_Q[i](f)
            K  = self.W_K[i](f)
            V  = self.W_V[i](f)
            sc = torch.bmm(Q, K.transpose(1,2)) / (self.d_j ** 0.5)
            h  = torch.bmm(F.softmax(sc, dim=-1), V)
            heads.append(h.reshape(B, S, N, -1))
        return self.W_MH(torch.cat(heads, dim=-1))

print('MAF defined (no diffusion).')

In [ ]:
# ══════════════════════════════════════════════════
# MGRC — dual adjacency + 3 stacked GCN+GRU layers
# ══════════════════════════════════════════════════

class MultiGraphFusion(nn.Module):
    def __init__(self, N):
        super().__init__()
        self.E1     = nn.Parameter(torch.randn(N, 1) * 0.01)
        self.E2     = nn.Parameter(torch.randn(N, 1) * 0.01)
        self.conv2d = nn.Conv2d(2, 1, kernel_size=1)

    def forward(self, A_dist):
        A_dyna = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        stack  = torch.stack([A_dyna, A_dist], dim=0).unsqueeze(0)
        A_F    = F.relu(self.conv2d(stack).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)


class GCN_GRU_Layer(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.W_GCN = nn.Linear(in_dim, hid_dim, bias=False)
        self.gru   = nn.GRUCell(hid_dim, hid_dim)

    def forward(self, x_seq, A_F):
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B*N, self.gru.hidden_size, device=x_seq.device)
        outs = []
        for t in range(S):
            agg = torch.einsum('nm,bmd->bnd', A_F, x_seq[:,t])
            xp  = F.relu(self.W_GCN(agg))
            h   = self.gru(xp.reshape(B*N,-1), h)
            outs.append(h.reshape(B,N,-1))
        return torch.stack(outs, dim=1)


class MGRCModule(nn.Module):
    def __init__(self, in_dim, hid_dim, N, n_layers=3):
        super().__init__()
        self.fusion = MultiGraphFusion(N)
        dims = [in_dim] + [hid_dim]*n_layers
        self.layers = nn.ModuleList([
            GCN_GRU_Layer(dims[i], dims[i+1]) for i in range(n_layers)])

    def forward(self, x, A_dist):
        A_F = self.fusion(A_dist)
        for layer in self.layers:
            x = layer(x, A_F)
        return x

print('MGRC defined (3 GCN+GRU layers, dual adjacency).')

In [ ]:
class SpatialTransformerLayer(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d,d*4), nn.ReLU(), nn.Linear(d*4,d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dr)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.reshape(B*S, N, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, S, N, d)


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d,d*4), nn.ReLU(), nn.Linear(d*4,d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dr)

    def forward(self, x):
        B, S, N, d = x.shape
        f = x.permute(0,2,1,3).reshape(B*N, S, d)
        h, _ = self.attn(f, f, f)
        h = self.norm1(f + self.drop(h))
        h = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print('Transformer layers defined (×5 each).')

In [ ]:
class MDGRTN_NoDiff(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        L, h, dr = cfg.num_layers, cfg.n_heads, cfg.dropout
        P, S = cfg.pred_len, cfg.seq_len

        self.maf  = MAFModule(F, d, h, n_seqs=1)
        self.mgrc = MGRCModule(d, d, N, n_layers=cfg.gru_layers)

        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, N, d) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, S, 1, d) * 0.02)

        self.spatial_layers  = nn.ModuleList([SpatialTransformerLayer(d, h, dr) for _ in range(L)])
        self.temporal_layers = nn.ModuleList([TemporalTransformerLayer(d, h, dr) for _ in range(L)])

        self.out_proj = nn.Linear(d * S, P)

    def forward(self, x, A_dist):
        # x: (B,S,N,F) → (B,P,N)
        x = self.maf([x])
        x = self.mgrc(x, A_dist)
        x = x + self.spatial_pos
        for layer in self.spatial_layers:
            x = layer(x)
        x = x + self.temporal_pos
        for layer in self.temporal_layers:
            x = layer(x)
        B, S, N, d = x.shape
        return self.out_proj(x.permute(0,2,1,3).reshape(B,N,S*d)).permute(0,2,1)


set_seed()
model = MDGRTN_NoDiff(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    out   = model(dummy, A_dist)
print(f'Output: {out.shape}  ✓  (expected [2, {cfg.pred_len}, {cfg.num_nodes}])')

In [ ]:
def mae_metric(pred, true):
    return torch.abs(pred - true).mean()

def rmse_metric(pred, true):
    return ((pred - true)**2).mean().sqrt()

def mape_metric(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

def huber_loss(pred, true, delta=1.0):
    return F.huber_loss(pred, true, delta=delta)

print('Metrics defined.')

In [ ]:
def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x, A_dist)
        loss = huber_loss(pred, y, cfg.delta_h)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        # denorm to real vehicle counts
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(mae_metric(pred_d, y_d).item())
        rmses.append(rmse_metric(pred_d, y_d).item())
        mapes.append(mape_metric(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train/eval functions defined.')

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_mae, history, cfg, drive_path=None):
    torch.save({
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch': epoch, 'best_mae': best_mae,
        'history': history, 'seed': SEED,
    }, cfg.ckpt_path)
    if drive_path:
        import shutil; shutil.copy(cfg.ckpt_path, drive_path)
    print(f'Ckpt ep={epoch} best_mae={best_mae:.3f}', end='\r')


def load_checkpoint(model, optimizer, scheduler, cfg, drive_path=None):
    if drive_path:
        import shutil; shutil.copy(drive_path, cfg.ckpt_path)
    if not os.path.exists(cfg.ckpt_path):
        print('No checkpoint — starting fresh.')
        return 1, float('inf'), {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}
    ckpt = torch.load(cfg.ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    print(f'Resumed epoch {ckpt["epoch"]} | best_mae={ckpt["best_mae"]:.3f}')
    return ckpt['epoch']+1, ckpt['best_mae'], ckpt['history']

print('Checkpoint utilities ready.')

In [ ]:
set_seed()

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=1e-6)

# ── To RESUME uncomment: ──
# start_epoch, best_val_mae, history = load_checkpoint(
#     model, optimizer, scheduler, cfg,
#     drive_path='/content/drive/MyDrive/mdgrtn_nodiff_ckpt.pt')

DRIVE_CKPT   = None   # set to Drive path to auto-save
start_epoch  = 1
best_val_mae = float('inf')
patience_cnt = 0
history      = {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}

print(f'Training | epochs={cfg.epochs} | patience={cfg.patience}')
print(f'No diffusion | MAF(1 seq) → MGRC(×{cfg.gru_layers}) → STFormer(×{cfg.num_layers})')
print(f'Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*65)

for epoch in range(start_epoch, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  ← best ✓'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
          f'MAE={val_mae:.3f}  RMSE={val_rmse:.3f}  MAPE={val_mape:.2f}%{tag}')

    save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_mae, history, cfg, drive_path=DRIVE_CKPT)

print(f'\nBest Val MAE: {best_val_mae:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', lw=2)
axes[0].set_title('Huber Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_mae'], color='orange', lw=2, label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', lw=1.5, label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='green', lw=2, label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', lw=1.5, label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('curves_nodiff.png', dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, A_dist, device, mean_flow, std_flow):
    """
    Paper-style evaluation:
    - Denorm predictions to real vehicle counts
    - Average MAE/RMSE/MAPE over ALL 12 steps and ALL 170 sensors
    - No error accumulation — direct one-shot prediction
    """
    model.eval()
    all_pred, all_true = [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)                               # (B,12,170) normalized
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]  # real units
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)   # (total_samples, 12, 170)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*58)
    print('  TEST RESULTS  (no diffusion, single sequence)')
    print('  Averaged over all 12 steps × 170 sensors')
    print('='*58)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*58)
    return mae, rmse, mape

paper_eval(model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(mae_metric(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(rmse_metric(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(mape_metric(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)